# Bronze Layer — Data Ingestion

The Bronze layer is the first stage in the Medallion Architecture. It preserves raw data from source systems with minimal transformation.

## Purpose

- **Ingest** raw CSV datasets using PySpark
- **Preserve** source data structure and grain (no aggregations, no joins, no filtering)
- **Add metadata** for data lineage (_ingested_at, _source)
- **Store** data in Parquet format for efficient access
- **Validate** that row counts match between raw and Bronze layers

## Architecture

Raw CSV → Bronze Parquet → Silver (cleaned) → Gold (business-ready) → ML/Analytics

Bronze is NOT cleaned or deduplicated. Business logic will be applied in Silver and Gold layers.

## Imports and Spark Session

In [10]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp, lit
from datetime import datetime
import os
import shutil

spark = SparkSession.builder \
    .appName("Olist-Bronze-Ingestion") \
    .getOrCreate()

print(f"Spark Session Started: {spark.sparkContext.appName}")

Spark Session Started: Olist-Bronze-Ingestion


## Pipeline Configuration

In [11]:
RAW_PATH = "../../data/raw"
BRONZE_PATH = "../../data/bronze"

SOURCE_FILES = {
    "customers": "olist_customers_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv"
}

print(f"Configured {len(SOURCE_FILES)} tables for ingestion")

Configured 8 tables for ingestion


## Raw Data Ingestion

In [12]:
bronze_dfs = {}
ingestion_timestamp = current_timestamp()

for table_name, filename in SOURCE_FILES.items():
    file_path = os.path.join(RAW_PATH, filename)
    
    df = spark.read.csv(file_path, header=True, inferSchema=True)
    bronze_dfs[table_name] = df
    
    row_count = df.count()
    col_count = len(df.columns)
    
    print(f"{table_name}: {row_count:,} rows, {col_count} columns")

customers: 99,441 rows, 5 columns
orders: 99,441 rows, 8 columns
order_items: 112,650 rows, 7 columns
order_payments: 103,886 rows, 5 columns
order_reviews: 104,162 rows, 7 columns
products: 32,951 rows, 9 columns
sellers: 3,095 rows, 4 columns
geolocation: 1,000,163 rows, 5 columns


## Schema Inspection

In [13]:
for table_name, df in bronze_dfs.items():
    print(f"\n{'='*60}")
    print(f"{table_name.upper()}")
    print(f"{'='*60}")
    df.printSchema()
    print(f"Rows: {df.count():,} | Columns: {len(df.columns)}")


CUSTOMERS
root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)

Rows: 99,441 | Columns: 5

ORDERS
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)

Rows: 99,441 | Columns: 8

ORDER_ITEMS
root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- p

## Add Bronze Metadata

In [14]:
bronze_dfs_with_metadata = {}

for table_name, df in bronze_dfs.items():
    df_with_metadata = df \
        .withColumn("_ingested_at", current_timestamp()) \
        .withColumn("_source", lit(SOURCE_FILES[table_name]))
    
    bronze_dfs_with_metadata[table_name] = df_with_metadata

print("Metadata added to all tables")

Metadata added to all tables


## Display Sample with Metadata

In [15]:
for table_name, df in list(bronze_dfs_with_metadata.items())[:2]:
    print(f"{table_name.upper()} (first row with metadata):")
    df.show(1, truncate=False)

CUSTOMERS (first row with metadata):
+--------------------------------+--------------------------------+------------------------+-------------+--------------+--------------------------+---------------------------+
|customer_id                     |customer_unique_id              |customer_zip_code_prefix|customer_city|customer_state|_ingested_at              |_source                    |
+--------------------------------+--------------------------------+------------------------+-------------+--------------+--------------------------+---------------------------+
|06b8999e2fba1a1fbc88172c00ba8bc7|861eff4711a542e4b93843c6dd7febb0|14409                   |franca       |SP            |2026-08-31 01:13:48.134957|olist_customers_dataset.csv|
+--------------------------------+--------------------------------+------------------------+-------------+--------------+--------------------------+---------------------------+
only showing top 1 row
ORDERS (first row with metadata):
+--------------------

## Write Bronze Data

In [ ]:
write_status = {}

for table_name, df in bronze_dfs_with_metadata.items():
    output_path = os.path.join(BRONZE_PATH, table_name)
    
    df.coalesce(1) \
        .write \
        .mode("overwrite") \
        .parquet(output_path)
    
    write_status[table_name] = output_path


## Pipeline Summary

### Ingestion Complete

**Bronze Layer Properties:**

- Source data is preserved as-is
- Duplicates NOT removed (preserved from source)
- Null values NOT removed (preserved from source)
- No business logic applied
- No derived columns created
- Ready for Silver layer transformations
